# Prétraitement et Extraction de Features

Ce notebook répond au skill `brainscan-feature-extraction`.
Il vise à transformer les images PNG/JPG en vecteurs numériques exploitables via `ResNet50` pré-entraîné sur `ImageNet`.

In [1]:
import os
import glob
import numpy as np
import torch
import torch.nn as nn
from torchvision import models, transforms
from torchvision.models import ResNet50_Weights
from PIL import Image
from tqdm.notebook import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu'))
print(f"🖥️  Utilisation de l'accélération matérielle: {device}")

🖥️  Utilisation de l'accélération matérielle: mps


In [2]:
# Chargement du modèle ResNet50 pré-entraîné et retrait de la couche Fully Connected (Classification Head)
print("⏳ Chargement du modèle ResNet50...")
weights = ResNet50_Weights.IMAGENET1K_V1
model = models.resnet50(weights=weights)
model = nn.Sequential(*list(model.children())[:-1])
model = model.to(device)
model.eval()

# Geler les couches convolutives
for param in model.parameters():
    param.requires_grad = False

⏳ Chargement du modèle ResNet50...


In [3]:
# Transformation pipeline (Redimensionnement et Normalisation ImageNet)
preprocess = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

In [4]:
# Initialisation des listes et itération sur les dossiers pour l'extraction
base_dir = "../data/raw/mri_dataset_brain_cancer_oc"
folders = {
    "Normal": os.path.join(base_dir, "avec_labels", "Normal"),
    "Cancer": os.path.join(base_dir, "avec_labels", "Cancer"),
    "non_labellise": os.path.join(base_dir, "sans_label")
}

extracted_features = []
extracted_filenames = []
extracted_labels = []

for label, folder in folders.items():
    if not os.path.exists(folder):
        continue
        
    image_paths = glob.glob(os.path.join(folder, "*.jpg")) + glob.glob(os.path.join(folder, "*.png"))
    print(f"📂 Catégorie [{label}] - {len(image_paths)} images trouvées")
    
    for img_path in tqdm(image_paths, desc=f"Extraction {label}"):
        try:
            img = Image.open(img_path).convert('RGB')
            img_tensor = preprocess(img).unsqueeze(0).to(device)
            
            with torch.no_grad():
                feature = model(img_tensor)
            
            feature = feature.view(-1).cpu().numpy()
            
            extracted_features.append(feature)
            extracted_filenames.append(os.path.basename(img_path))
            extracted_labels.append(label)
            
        except Exception as e:
            # Ignore 
            pass

📂 Catégorie [Normal] - 50 images trouvées


Extraction Normal:   0%|          | 0/50 [00:00<?, ?it/s]

📂 Catégorie [Cancer] - 50 images trouvées


Extraction Cancer:   0%|          | 0/50 [00:00<?, ?it/s]

📂 Catégorie [non_labellise] - 1406 images trouvées


Extraction non_labellise:   0%|          | 0/1406 [00:00<?, ?it/s]

In [5]:
# Sauvegarde dans un fichier performant compressé (.npy)
output_dir = "../data"
os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(output_dir, "features_brainscan.npy")

features_array = np.array(extracted_features)
filenames_array = np.array(extracted_filenames)
labels_array = np.array(extracted_labels)

if len(features_array) > 0:
    np.save(output_path, {
        "features": features_array,
        "filenames": filenames_array,
        "labels": labels_array
    }, allow_pickle=True)
    print(f"✅ {len(features_array)} embeddings sauvegardés avec succès.")
    print(f"Shape des features : {features_array.shape}")
else:
    print("⚠️ Aucune feature extraite.")

✅ 1506 embeddings sauvegardés avec succès.
Shape des features : (1506, 2048)
